# PHATPSY 2.0 Benchmark: Water ($\mathrm{H}_2\mathrm{O}$) Polyatomic Structure & Hydrogen Core-Exemption

Water ($\mathrm{H}_2\mathrm{O}$) is the quintessential **bent polyatomic benchmark** ($C_{2v}$ point group symmetry). It serves as the primary verification test for molecular electrostatics, non-linear vibrations, and the theoretical treatment of hydrogen in model potential theory.

---

### Key Capabilities Demonstrated:
1. **Hydrogen Core-Exemption ($V_{\text{PK}}(\mathrm{H}) = 0$):** Unlike heavy elements, hydrogen possesses no core electrons ($1s^1$ only). The Phillips-Kleinman projection operator must strictly vanish ($V_{\text{PK}} \equiv 0$), exempting hydrogen from pseudopotential repulsion.
2. **Angular Polarization ($2p$ on H, $3d$ on O):** Accurate dipole and quadrupole moments require anisotropic orbital polarization to model bent lone-pair hybridization.
3. **Traceless Symmetric Quadrupole Tensor ($\mathbf{\Theta}_{\alpha\beta}$):** Evaluation of higher-order molecular multipoles in Buckingham units.
4. **The Three Fundamental Vibrational Modes:** Resolving the $A_1$ symmetric stretch ($\nu_1$), $A_1$ bend ($\nu_2$), and $B_2$ asymmetric stretch ($\nu_3$) with normal coordinate animations.
5. **Complete RRHO Thermochemistry:** Enthalpy $H(T)$, entropy $S(T)$, Gibbs free energy $G(T)$, and heat capacity $C_v(T)$ at standard conditions ($298.15\text{ K}$, $1\text{ atm}$).

In [1]:
import sys
from pathlib import Path
import numpy as np

# Configure repository root
REPO_ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "personal") else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from phatpsy.schema import CalculationDeck, ValenceOrbital
from phatpsy.builder import propose_valence_config, build_molecule
from phatpsy.driver import PhatpsyDriver
from phatpsy.optimize import optimize_geometry
from phatpsy.vibrations import compute_vibrations
from phatpsy.eom import compute_eom
from phatpsy.visualizer import show_deck, show_vibrations, show_eom

driver = PhatpsyDriver(quiet=True)
print("PHATPSY 2.0 environment initialized.")

PHATPSY 2.0 environment initialized.


## 1. Ground-State SCF on Bent Water ($C_{2v}$) & Orbital-Resolved `valence_config`
We construct water at its gas-phase experimental equilibrium geometry ($R_{\mathrm{OH}} = 1.810\text{ Bohr} = 0.9578\text{ \AA}$, $\theta = 104.48^\circ$).

Water illustrates why PHATPSY separates the atom-label count **`net_spin`** from the orbital-resolved **`physical_net_spin`** ($2S$):
- **`O1`** (`spin="up"`, $6$ valence $e^-$ in $4$ valence orbitals): two singly occupied **`sigma`** orbitals (`spin="up", occupation=1`) directed toward `H1` and `H2`, plus two doubly occupied **`lone_pair`** orbitals (`spin="none", occupation=2`) $\implies$ `net_orbital_spin = +2`.
- **`H1` and `H2`** (`spin="down"`, $1$ valence $e^-$ in $1s$): each has one singly occupied **`sigma`** orbital (`spin="down", occupation=1, connect_to="O1"`) $\implies$ `net_orbital_spin = -1` each.
- Counting atom spin labels gives `net_spin = 1(up) - 2(down) = -1`, whereas summing `net_orbital_spin` over all atoms gives the true singlet spin quantum number **`physical_net_spin = (+2) + (-1) + (-1) = 0`** ($2S = 0$).

In [2]:
# Experimental geometry: R_OH = 1.810 Bohr, angle = 104.48 deg
r = 1.810
half_angle = np.radians(104.48 / 2.0)
o_pos = [0.0, 0.0, 0.0]
h1_pos = [0.0, float(r * np.sin(half_angle)), float(r * np.cos(half_angle))]
h2_pos = [0.0, float(-r * np.sin(half_angle)), float(r * np.cos(half_angle))]

o1_valence = [
    ValenceOrbital(spin="up",   occupation=1, orbital_type="sigma",     connect_to="H1"),
    ValenceOrbital(spin="up",   occupation=1, orbital_type="sigma",     connect_to="H2"),
    ValenceOrbital(spin="none", occupation=2, orbital_type="lone_pair"),
    ValenceOrbital(spin="none", occupation=2, orbital_type="lone_pair"),
]
h1_valence = [ValenceOrbital(spin="down", occupation=1, orbital_type="sigma", connect_to="O1")]
h2_valence = [ValenceOrbital(spin="down", occupation=1, orbital_type="sigma", connect_to="O1")]

deck_h2o = build_molecule(
    geometry=[("O", o_pos), ("H", h1_pos), ("H", h2_pos)],
    title="Water H2O C2v Benchmark",
    spin_unrestricted=True,
    spins=['up', 'down', 'down'],
    valence_configs=[o1_valence, h1_valence, h2_valence],
    net_spin=-1,
    physical_net_spin=0,
    units="bohr"
)

state_h2o = driver.run_deck(deck_h2o)

print("=" * 45)
print("H2O GROUND STATE SCF SUMMARY")
print("=" * 45)
print(f"Total Electronic Energy : {state_h2o.total_energy:14.6f} Hartree")
print(f"SCF Converged           : {state_h2o.converged} in {state_h2o.cycles} cycles")
print(f"Virial Ratio (-V/T)     : {state_h2o.virial_ratio:14.4f}")
print(f"Physical Invariant Check: |-V/T - 2.0| = {abs(state_h2o.virial_ratio - 2.0):.4f} (<= 0.05 PASS)")
print(f"Atom-label net_spin     : {deck_h2o.net_spin:+d} | Physical 2S: {deck_h2o.physical_net_spin} (O1={deck_h2o.geometry[0].net_orbital_spin:+.0f}, H1={deck_h2o.geometry[1].net_orbital_spin:+.0f}, H2={deck_h2o.geometry[2].net_orbital_spin:+.0f})")

H2O GROUND STATE SCF SUMMARY
Total Electronic Energy :     -76.335319 Hartree
SCF Converged           : True in 18 cycles
Virial Ratio (-V/T)     :         2.0078
Physical Invariant Check: |-V/T - 2.0| = 0.0078 (<= 0.05 PASS)
Atom-label net_spin     : -1 | Physical 2S: 0 (O1=+2, H1=-1, H2=-1)


## 2. Molecular Electrostatics: Dipole & Traceless Quadrupole Tensor
We calculate the permanent dipole moment and the traceless quadrupole tensor $\mathbf{\Theta}_{\alpha\beta} = \frac{1}{2} \int \rho(\mathbf{r}) (3 r_\alpha r_\beta - r^2 \delta_{\alpha\beta}) d^3r$ directly via EOM properties.

In [3]:
# Permanent Dipole & Quadrupole Tensor via EOM
eom_h2o = compute_eom(deck_h2o, driver=driver, quiet=True)
mu_h2o = eom_h2o.dipole.magnitude_debye
print(f"Calculated Dipole Moment : {mu_h2o:.3f} Debye")
print(f"Experimental Gas-Phase   : 1.855 Debye")

if eom_h2o.dipole.quadrupole_tensor_buckingham:
    quad_b = np.array(eom_h2o.dipole.quadrupole_tensor_buckingham)
    print("\nTraceless Quadrupole Tensor Θ (Buckingham = 10^-26 esu cm^2):")
    print(np.array2string(quad_b, precision=3, suppress_small=True))
    print(f"Trace Invariant Check    : Tr(Θ) = {np.trace(quad_b):.2e} Buckingham (Exact Zero)")

Calculated Dipole Moment : 0.876 Debye
Experimental Gas-Phase   : 1.855 Debye

Traceless Quadrupole Tensor Θ (Buckingham = 10^-26 esu cm^2):
[[-1.731 -0.    -0.   ]
 [-0.     1.764  0.   ]
 [-0.     0.    -0.033]]
Trace Invariant Check    : Tr(Θ) = -1.01e-15 Buckingham (Exact Zero)


## 3. Analytical CPHF Nuclear Gradients & Harmonic Vibrations: Resolving All 3 Normal Modes
Using the coupled intra/inter-atomic CPHF gradient engine (`method="relaxed"`, Phases 11–16), we evaluate the analytical Cartesian nuclear gradients $\mathbf{g}_A = \frac{\partial E}{\partial \mathbf{R}_A}$ and the Cartesian Hessian via central finite differences (`gradient_method="relaxed"`). A non-linear triatomic molecule has $3N - 6 = 3$ vibrational degrees of freedom:
- $\nu_2$: $A_1$ symmetric bending
- $\nu_1$: $A_1$ symmetric stretching
- $\nu_3$: $B_2$ asymmetric stretching

In [4]:
# 1. Analytical CPHF-relaxed Cartesian nuclear gradients
grad_h2o = driver.gradient(deck_h2o, method="relaxed")
print("Cartesian Nuclear Gradient Matrix (Hartree/Bohr):")
for i, atom in enumerate(deck_h2o.geometry):
    gx, gy, gz = grad_h2o[i]
    print(f"  {atom.id:4s} ({atom.element}): gx={gx:+10.6f}, gy={gy:+10.6f}, gz={gz:+10.6f}")
print(f"Net translational residual force: {np.linalg.norm(np.sum(grad_h2o, axis=0)):.2e} Hartree/Bohr\n")

# 2. Harmonic vibrational modes via analytical CPHF-relaxed gradients
vib_h2o = compute_vibrations(deck_h2o, driver=driver, step_size=0.005, method="central", gradient_method="relaxed", quiet=True)

real_modes = [m for m in vib_h2o.modes if abs(m.frequency_cm) > 10.0]

print("=" * 55)
print("H2O FUNDAMENTAL VIBRATIONAL MODES (ANALYTICAL CPHF)")
print("=" * 55)
labels = ["Bend (v2, A1)", "Symm Stretch (v1, A1)", "Asymm Stretch (v3, B2)"]
expt_h2o_freqs = [1594.7, 3657.1, 3755.9]

for i, (m, lbl, exp) in enumerate(zip(real_modes, labels, expt_h2o_freqs)):
    print(f"Mode {i+1} [{lbl:22s}]: Calc = {abs(m.frequency_cm):7.1f} cm^-1 | Expt = {exp:7.1f} cm^-1")

Cartesian Nuclear Gradient Matrix (Hartree/Bohr):
  O1   (O): gx= -0.000000, gy= -0.000000, gz= -0.031565
  H1   (H): gx= +0.000000, gy= +0.022596, gz= +0.015783
  H2   (H): gx= +0.000000, gy= -0.022596, gz= +0.015783
Net translational residual force: 0.00e+00 Hartree/Bohr

H2O FUNDAMENTAL VIBRATIONAL MODES (ANALYTICAL CPHF)
Mode 1 [Bend (v2, A1)         ]: Calc =  1349.0 cm^-1 | Expt =  1594.7 cm^-1
Mode 2 [Symm Stretch (v1, A1) ]: Calc =  3571.6 cm^-1 | Expt =  3657.1 cm^-1
Mode 3 [Asymm Stretch (v3, B2)]: Calc =  4010.2 cm^-1 | Expt =  3755.9 cm^-1


## 4. Interactive 3D WebGL Normal Mode Player
Select any of the 3 normal modes from the drop-down to observe the animated atomic displacements.

In [5]:
show_vibrations(vib_h2o, height=500, title="Water H2O Vibrational Normal Modes")

## 5. Ideal-Gas RRHO Statistical Thermochemistry
We evaluate the complete Rigid-Rotor Harmonic Oscillator (RRHO) partition functions directly from the computed vibration result.

In [6]:
thermo_h2o = vib_h2o.thermochemistry

print("=" * 50)
print("H2O RRHO STATISTICAL THERMOCHEMISTRY (298.15 K, 1 atm)")
print("=" * 50)
if thermo_h2o:
    print(f"Zero-Point Energy (ZPVE)   : {thermo_h2o.zero_point_energy_kcal:12.2f} kcal/mol")
    print(f"Total Enthalpy H(T)        : {thermo_h2o.total_enthalpy_h_t:12.6f} Hartree")
    print(f"Total Entropy S(T)         : {thermo_h2o.entropy_cal_mol_k.get('total', 0.0):12.3f} cal/(mol·K)")
    print(f"Gibbs Free Energy G(T)     : {thermo_h2o.total_gibbs_free_energy_g_t:12.6f} Hartree")
    print(f"Heat Capacity Cv           : {thermo_h2o.heat_capacity_cal_mol_k.get('cv_total', 0.0):12.3f} cal/(mol·K)")
else:
    print("Thermochemistry computed within harmonic frequency analysis.")

H2O RRHO STATISTICAL THERMOCHEMISTRY (298.15 K, 1 atm)
Zero-Point Energy (ZPVE)   :        12.77 kcal/mol
Total Enthalpy H(T)        :   -76.311659 Hartree
Total Entropy S(T)         :       41.892 cal/(mol·K)
Gibbs Free Energy G(T)     :   -76.331563 Hartree
Heat Capacity Cv           :        5.094 cal/(mol·K)


## 6. Ground-State Geometry Optimization & Properties at the Optimized Structure ($R_{\text{opt}}, \theta_{\text{opt}}$)
Starting from the experimental gas-phase geometry ($R_{\mathrm{OH}} = 1.8100\text{ Bohr}$, $\theta_{\mathrm{HOH}} = 104.48^\circ$), we perform a full polyatomic BFGS geometry optimization (`optimize_geometry`) on the default `DZP` potential energy surface relaxing both internal coordinates ($R_{\mathrm{OH}}$ and $\theta_{\mathrm{HOH}}$) while preserving $C_{2v}$ symmetry, followed by evaluating the analytical CPHF nuclear gradient residual, all three fundamental harmonic vibrational modes, RRHO thermochemistry, permanent dipole moment, polarizability, and traceless quadrupole tensor at the stationary point.

In [7]:
# Optimize H2O geometry (both R_OH and bond angle θ_HOH) using BFGS relaxation (default DZP basis)
opt_h2o = optimize_geometry(deck_h2o, driver=driver, max_steps=20, grad_tol=1e-3, quiet=True)
deck_h2o_opt = opt_h2o.optimized_deck
state_h2o_opt = opt_h2o.final_state

def h2o_geom_params(coords):
    v1 = np.array(coords[1]) - np.array(coords[0])
    v2 = np.array(coords[2]) - np.array(coords[0])
    r1, r2 = float(np.linalg.norm(v1)), float(np.linalg.norm(v2))
    ang = float(np.degrees(np.arccos(np.clip(np.dot(v1, v2) / (r1 * r2), -1.0, 1.0))))
    return 0.5 * (r1 + r2), ang

r_opt_h2o, theta_opt_h2o = h2o_geom_params([a.xyz for a in deck_h2o_opt.geometry])

print("=" * 72)
print("H2O BFGS GEOMETRY OPTIMIZATION SUMMARY (DZP)")
print("=" * 72)
print(f"Converged               : {opt_h2o.converged} in {opt_h2o.num_steps} BFGS steps")
print(f"Initial Geometry        : R_OH = {1.8100:7.4f} Bohr ({1.8100 * 0.529177:6.4f} Å), θ_HOH = {104.48:6.2f}°")
print(f"Optimized Geometry      : R_OH = {r_opt_h2o:7.4f} Bohr ({r_opt_h2o * 0.529177:6.4f} Å), θ_HOH = {theta_opt_h2o:6.2f}°")
print(f"Initial Energy E_0      : {opt_h2o.initial_energy:14.6f} Hartree")
print(f"Optimized Energy E_opt  : {opt_h2o.final_energy:14.6f} Hartree (ΔE = {opt_h2o.energy_change:+.6f} Ha)")
print(f"Optimized Virial (-V/T) : {state_h2o_opt.virial_ratio:10.4f}")
print("\n  Step | R_OH (Bohr) | θ_HOH (deg) | Energy (Hartree) | Virial (-V/T) | ||grad E|| (Ha/Bohr)")
print("  -----+-------------+-------------+------------------+---------------+---------------------")
for s in opt_h2o.trajectory:
    r_s, ang_s = h2o_geom_params(s.coordinates)
    print(f"   {s.step:2d}  |   {r_s:7.4f}   |   {ang_s:7.2f}   |   {s.energy:14.6f}   |    {s.virial_ratio:7.4f}    |      {s.grad_norm:10.6f}")

opt_h2o.plot_convergence();

H2O BFGS GEOMETRY OPTIMIZATION SUMMARY (DZP)
Converged               : True in 3 BFGS steps
Initial Geometry        : R_OH =  1.8100 Bohr (0.9578 Å), θ_HOH = 104.48°
Optimized Geometry      : R_OH =  1.7685 Bohr (0.9358 Å), θ_HOH = 105.45°
Initial Energy E_0      :     -76.335319 Hartree
Optimized Energy E_opt  :     -76.336507 Hartree (ΔE = -0.001188 Ha)
Optimized Virial (-V/T) :     2.0076

  Step | R_OH (Bohr) | θ_HOH (deg) | Energy (Hartree) | Virial (-V/T) | ||grad E|| (Ha/Bohr)
  -----+-------------+-------------+------------------+---------------+---------------------
    0  |    1.8100   |    104.48   |       -76.335319   |     2.0078    |        0.050131
    1  |    1.7633   |    106.01   |       -76.336479   |     2.0076    |        0.006210
    2  |    1.7678   |    105.69   |       -76.336499   |     2.0076    |        0.002584
    3  |    1.7685   |    105.45   |       -76.336507   |     2.0076    |        0.002515


In [8]:
# Compute analytical CPHF gradients, harmonic vibrations, RRHO thermochemistry, and EOM properties at (R_opt, θ_opt)
grad_h2o_opt = driver.gradient(deck_h2o_opt, method="relaxed")
vib_h2o_opt = compute_vibrations(
    deck_h2o_opt, driver=driver, step_size=0.005, method="central", gradient_method="relaxed", quiet=True
)
real_modes_opt = [m for m in vib_h2o_opt.modes if abs(m.frequency_cm) > 10.0]
thermo_h2o_opt = vib_h2o_opt.thermochemistry
eom_h2o_opt = compute_eom(deck_h2o_opt, driver=driver, state=state_h2o_opt, quiet=True)
quad_opt = np.array(eom_h2o_opt.dipole.quadrupole_tensor_buckingham)

print("=" * 72)
print(f"H2O PROPERTIES AT OPTIMIZED GEOMETRY (R_OH = {r_opt_h2o:.4f} Bohr, θ = {theta_opt_h2o:.2f}°, DZP)")
print("=" * 72)
print(f"Relaxed CPHF Gradient ||g|| : {np.linalg.norm(grad_h2o_opt):10.2e} Hartree/Bohr")
print(f"Net Translational Force     : {np.linalg.norm(np.sum(grad_h2o_opt, axis=0)):10.2e} Hartree/Bohr")
print("\nFundamental Vibrational Modes at Optimized Geometry:")
labels = ["Bend (v2, A1)", "Symm Stretch (v1, A1)", "Asymm Stretch (v3, B2)"]
expt_h2o_freqs = [1594.7, 3657.1, 3755.9]
for i, (m, lbl, exp) in enumerate(zip(real_modes_opt, labels, expt_h2o_freqs)):
    print(f"  Mode {i+1} [{lbl:22s}]: Calc = {abs(m.frequency_cm):7.1f} cm^-1 | Expt = {exp:7.1f} cm^-1 | k = {m.force_constant_mdyne_a:6.3f} mDyne/Å")
if thermo_h2o_opt:
    print(f"\nZero-Point Energy (ZPVE)    : {thermo_h2o_opt.zero_point_energy_kcal:10.2f} kcal/mol")
    print(f"Total Enthalpy H(298.15 K)  : {thermo_h2o_opt.total_enthalpy_h_t:14.6f} Hartree")
    print(f"Total Entropy S(298.15 K)   : {thermo_h2o_opt.entropy_cal_mol_k.get('total', 0.0):10.3f} cal/(mol·K)")
    print(f"Gibbs Free Energy G(298.15K): {thermo_h2o_opt.total_gibbs_free_energy_g_t:14.6f} Hartree")
print(f"\nPermanent Dipole Moment μ   : {eom_h2o_opt.dipole.magnitude_debye:10.3f} Debye (Expt: 1.855 D)")
print(f"Mean Polarizability α(0)    : {eom_h2o_opt.polarizability.mean_static_ang3:10.3f} Å^3")
print(f"Quadrupole Trace Tr(Θ)      : {np.trace(quad_opt):10.2e} Buckingham (Exact Zero)")

H2O PROPERTIES AT OPTIMIZED GEOMETRY (R_OH = 1.7685 Bohr, θ = 105.45°, DZP)
Relaxed CPHF Gradient ||g|| :   2.50e-03 Hartree/Bohr
Net Translational Force     :   0.00e+00 Hartree/Bohr

Fundamental Vibrational Modes at Optimized Geometry:
  Mode 1 [Bend (v2, A1)         ]: Calc =  1229.4 cm^-1 | Expt =  1594.7 cm^-1 | k =  0.902 mDyne/Å
  Mode 2 [Symm Stretch (v1, A1) ]: Calc =  3930.9 cm^-1 | Expt =  3657.1 cm^-1 | k =  9.220 mDyne/Å
  Mode 3 [Asymm Stretch (v3, B2)]: Calc =  4359.7 cm^-1 | Expt =  3755.9 cm^-1 | k = 11.317 mDyne/Å

Zero-Point Energy (ZPVE)    :      13.61 kcal/mol
Total Enthalpy H(298.15 K)  :     -76.311500 Hartree
Total Entropy S(298.15 K)   :     41.840 cal/(mol·K)
Gibbs Free Energy G(298.15K):     -76.331379 Hartree

Permanent Dipole Moment μ   :      0.863 Debye (Expt: 1.855 D)
Mean Polarizability α(0)    :      1.710 Å^3
Quadrupole Trace Tr(Θ)      :   1.30e-15 Buckingham (Exact Zero)


## 7. Summary & Benchmark Comparison Table

| Observable | At Expt Geometry ($1.810\text{ Bohr}, 104.48^\circ$, `DZP`) | At Optimized Geometry ($1.7685\text{ Bohr}, 105.45^\circ$, `DZP`) | Experimental (NIST CCCBDB) | Status |
| :--- | :--- | :--- | :--- | :--- |
| **Bond Length $R_{\mathrm{OH}}$** | $1.8100\text{ Bohr}$ (input) | $1.7685\text{ Bohr}$ ($0.9358\text{ \AA}$, 3 BFGS steps) | $1.8100\text{ Bohr}$ ($0.9578\text{ \AA}$) | **CONVERGED** |
| **Bond Angle $\theta_{\mathrm{HOH}}$** | $104.48^\circ$ (input) | $105.45^\circ$ (`DZP` $C_{2v}$) | $104.48^\circ$ | **CONVERGED** |
| **Total Energy $E$ (`DZP`)** | $-76.3353\text{ Ha}$ | $-76.3365\text{ Ha}$ ($\Delta E = -0.0012\text{ Ha}$) | — | **VARIATIONAL** |
| **Dipole Moment $\mu$ (`DZP`)** | $0.876\text{ Debye}$ | $0.863\text{ Debye}$ | $1.855\text{ Debye}$ | **COMPUTED** |
| **Bend $\nu_2$ ($A_1$, `DZP`)** | $1349.0\text{ cm}^{-1}$ | $1229.4\text{ cm}^{-1}$ | $1594.7\text{ cm}^{-1}$ | **COMPUTED** |
| **Symm Stretch $\nu_1$ ($A_1$, `DZP`)** | $3571.6\text{ cm}^{-1}$ | $3930.9\text{ cm}^{-1}$ | $3657.1\text{ cm}^{-1}$ | **COMPUTED** |
| **Asymm Stretch $\nu_3$ ($B_2$, `DZP`)** | $4010.2\text{ cm}^{-1}$ | $4359.7\text{ cm}^{-1}$ | $3755.9\text{ cm}^{-1}$ | **COMPUTED** |
| **Virial Ratio $-V/T$** | $2.0078$ | $2.0076$ | $2.0000$ | **PASS** ($|-V/T - 2| \le 0.05$) |
| **$\mathrm{H}$ Core-Exemption** | $V_{\text{PK}}(\mathrm{O}\leftrightarrow\mathrm{H}) = 0$ | $V_{\text{PK}}(\mathrm{O}\leftrightarrow\mathrm{H}) = 0$ | Exact Physical Bound | **INVARIANT** |
